In [2]:
# ==============================================
# SILVER LAYER - Data Cleaning & Transformation
# ==============================================
# Purpose: Clean, standardize and enrich Bronze data
# Reads from: bronze/ (Parquet files)
# Writes to: silver/ (Parquet files)
# No aggregations here — that's Gold layer's job
# ==============================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Ecommerce-Silver-Layer") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Started Successfully")

Spark Session Started Successfully


In [3]:
import os

BRONZE_PATH = "bronze/"
SILVER_PATH = "silver/"

os.makedirs(SILVER_PATH, exist_ok=True)

# Read all bronze tables
orders         = spark.read.parquet("bronze/orders")
order_items    = spark.read.parquet("bronze/order_items")
customers      = spark.read.parquet("bronze/customers")
products       = spark.read.parquet("bronze/products")
sellers        = spark.read.parquet("bronze/sellers")
order_payments = spark.read.parquet("bronze/order_payments")
order_reviews  = spark.read.parquet("bronze/order_reviews")
geolocation    = spark.read.parquet("bronze/geolocation")
category_translation = spark.read.parquet("bronze/category_translation")

print("All bronze tables loaded successfully")

All bronze tables loaded successfully


In [4]:
def profile_table(df, table_name):
    print(f"\n{'='*50}")
    print(f"TABLE: {table_name.upper()}")
    print(f"{'='*50}")
    
    # Basic shape
    row_count = df.count()
    col_count = len(df.columns)
    print(f"Rows: {row_count} | Columns: {col_count}")
    
    # Schema first — types drive all transformation decisions
    print(f"\nSchema:")
    df.printSchema()
    
    # Null counts per column
    print(f"Null counts:")
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ])
    null_counts.show(truncate=False)
    
    # Duplicate row count
    duplicate_count = row_count - df.dropDuplicates().count()
    print(f"Duplicate rows: {duplicate_count}")
    
    # Sample records
    print(f"\nSample records (3 rows):")
    df.show(3, truncate=True)

In [4]:
profile_table(orders, "orders")


TABLE: ORDERS
Rows: 99441 | Columns: 8

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Null counts:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

In [5]:
profile_table(order_items, "order_items")


TABLE: ORDER_ITEMS
Rows: 112650 | Columns: 7

Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Null counts:
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|0       |0            |0         |0        |0                  |0    |0            |
+--------+-------------+----------+---------+-------------------+-----+-------------+

Duplicate rows: 0

Sample records (3 rows):
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|orde

In [6]:
orders.groupBy("order_status").agg(
    F.count(F.when(F.col("order_delivered_customer_date").isNull(), True))
    .alias("null_delivery_dates")
).show()

+------------+-------------------+
|order_status|null_delivery_dates|
+------------+-------------------+
|     shipped|               1107|
|    canceled|                619|
|    approved|                  2|
|    invoiced|                314|
|   delivered|                  8|
| unavailable|                609|
|  processing|                301|
|     created|                  5|
+------------+-------------------+



In [7]:
orders.filter(
    (F.col("order_status") == "delivered") &
    (F.col("order_delivered_customer_date").isNull())
).show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e69f75a717d64fc5ecdfae42b2e8e086|cfda40ca8dd0a5d486a9635b611b398a|delivered   |2018-07-01 22:05:55     |2018-07-01 22:15:14|2018-07-03 13:57:00         |NULL                         |2018-07-30 00:00:00          |
|0d3268bad9b086af767785e3f0fc0133|4f1d63d35fb7c8999853b2699f5c7649|delivered   |2018-07-01 21:14:02     |2018-07-01 21:29:54|2018-07-03 09:2

In [8]:
from pyspark.sql import functions as F

# Identify the anomaly:
# Orders marked as delivered but missing delivery timestamp
silver_orders = orders.withColumn(
    "is_anomaly",
    F.when(
        (F.col("order_status") == "delivered") & 
        (F.col("order_delivered_customer_date").isNull()),
        True
    ).otherwise(False)
)

# Verify our flag worked
print("Anomaly breakdown:")
silver_orders.groupBy("is_anomaly").count().show()

print("Sample anomalous records:")
silver_orders.filter(F.col("is_anomaly") == True).show(5, truncate=True)

Anomaly breakdown:
+----------+-----+
|is_anomaly|count|
+----------+-----+
|      true|    8|
|     false|99433|
+----------+-----+

Sample anomalous records:
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+----------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|is_anomaly|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+----------+
|e69f75a717d64fc5e...|cfda40ca8dd0a5d48...|   delivered|     2018-07-01 22:05:55|2018-07-01 22:15:14|         2018-07-03 13:57:00|                         NULL|          2018-07-30 00:00:00|      true|
|0d3268bad9b086af7...|4f1d63d35f

In [9]:
silver_orders.write.mode("overwrite").parquet("silver/orders")
print(f"Silver orders saved.")
print(f"Total rows: {silver_orders.count()}")
print(f"Anomalies flagged: {silver_orders.filter(F.col('is_anomaly')==True).count()}")

Silver orders saved.
Total rows: 99441
Anomalies flagged: 8


In [10]:
profile_table(order_items, "order_items")


TABLE: ORDER_ITEMS
Rows: 112650 | Columns: 7

Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Null counts:
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|0       |0            |0         |0        |0                  |0    |0            |
+--------+-------------+----------+---------+-------------------+-----+-------------+

Duplicate rows: 0

Sample records (3 rows):
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|orde

In [11]:
# order_items — No transformation required
# Profiling showed: 0 nulls, 0 duplicate rows, 
# correct types, valid item-level granularity
# Preserving as-is from Bronze

silver_order_items = order_items

silver_order_items.write.mode("overwrite").parquet("silver/order_items")
print(f"Silver order_items saved: {silver_order_items.count()} rows")

Silver order_items saved: 112650 rows


In [12]:
profile_table(customers, "customers")



TABLE: CUSTOMERS
Rows: 99441 | Columns: 5

Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

Null counts:
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|0          |0                 |0                       |0            |0             |
+-----------+------------------+------------------------+-------------+--------------+

Duplicate rows: 0

Sample records (3 rows):
+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state

In [13]:
from pyspark.sql.functions import countDistinct

customers.select(
    countDistinct("customer_id").alias("unique_order_customers"),
    countDistinct("customer_unique_id").alias("unique_real_customers")
).show()

+----------------------+---------------------+
|unique_order_customers|unique_real_customers|
+----------------------+---------------------+
|                 99441|                96096|
+----------------------+---------------------+



In [14]:
# customers — No transformation required
# Profiling showed: 0 nulls, 0 duplicate rows
# Key insight: customer_id is order-scoped, 
# customer_unique_id is the true unique customer
# 96,096 unique real customers across 99,441 orders
# meaning 3,345 customers placed repeat orders

silver_customers = customers
silver_customers.write.mode("overwrite").parquet("silver/customers")
print(f"Silver customers saved: {silver_customers.count()} rows")

Silver customers saved: 99441 rows


In [15]:
profile_table(products, "products")


TABLE: PRODUCTS
Rows: 32951 | Columns: 9

Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

Null counts:
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+----------------

In [16]:
# Are the same 610 rows missing all values?
products.filter(
    F.col("product_category_name").isNull()
).select(
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
).show(5)

+--------------------+---------------------+-------------------+--------------------------+------------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|
+--------------------+---------------------+-------------------+--------------------------+------------------+
|a41e356c76fab6633...|                 NULL|               NULL|                      NULL|              NULL|
|d8dee61c2034d6d07...|                 NULL|               NULL|                      NULL|              NULL|
|56139431d72cd51f1...|                 NULL|               NULL|                      NULL|              NULL|
|46b48281eb6d663ce...|                 NULL|               NULL|                      NULL|              NULL|
|5fb61f482620cb672...|                 NULL|               NULL|                      NULL|              NULL|
+--------------------+---------------------+-------------------+--------------------------+------------------+
o

In [17]:
products.filter(
    F.col("product_category_name").isNull()
).show(10, truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|a41e356c76fab66334f36de622ecbd3a|NULL                 |NULL               |NULL                      |NULL              |650             |17               |14               |12              |
|d8dee61c2034d6d075997acef1870e9b|NULL                 |NULL               |NULL                      |NULL              |300             |16               |7                |20              |
|56139431d72cd51f19eb9f7dae4d1617|N

In [18]:
products.filter(
    F.col("product_category_name").isNull()
).select(
    F.count("*").alias("rows"),
    F.countDistinct("product_id").alias("distinct_products")
).show()

+----+-----------------+
|rows|distinct_products|
+----+-----------------+
| 610|              610|
+----+-----------------+



In [19]:
order_items.join(
    products.filter(F.col("product_category_name").isNull()),
    on="product_id",
    how="inner"
).count()

1603

In [20]:
# Step 1 — Fill missing category with 'Unknown'
silver_products = products.withColumn(
    "product_category_name",
    F.when(
        F.col("product_category_name").isNull(),
        "unknown"
    ).otherwise(F.col("product_category_name"))
)

# Step 2 — Join with translation table to get English categories
silver_products = silver_products.join(
    category_translation,
    on="product_category_name",
    how="left"
)

# Step 3 — Verify translation worked
silver_products.select(
    "product_id",
    "product_category_name",
    "product_category_name_english"
).show(5)

+--------------------+---------------------+-----------------------------+
|          product_id|product_category_name|product_category_name_english|
+--------------------+---------------------+-----------------------------+
|1e9e8ef04dbcff454...|           perfumaria|                    perfumery|
|3aa071139cb16b67c...|                artes|                          art|
|96bd76ec8810374ed...|        esporte_lazer|               sports_leisure|
|cef67bcfe19066a93...|                bebes|                         baby|
|9dc1a7de274444849...| utilidades_domest...|                   housewares|
+--------------------+---------------------+-----------------------------+
only showing top 5 rows


In [21]:
silver_products.filter(
    F.col("product_category_name") == "unknown"
).select(
    "product_id",
    "product_category_name",
    "product_category_name_english"
).show(5)

+--------------------+---------------------+-----------------------------+
|          product_id|product_category_name|product_category_name_english|
+--------------------+---------------------+-----------------------------+
|a41e356c76fab6633...|              unknown|                         NULL|
|d8dee61c2034d6d07...|              unknown|                         NULL|
|56139431d72cd51f1...|              unknown|                         NULL|
|46b48281eb6d663ce...|              unknown|                         NULL|
|5fb61f482620cb672...|              unknown|                         NULL|
+--------------------+---------------------+-----------------------------+
only showing top 5 rows


In [22]:
silver_products = silver_products.withColumnRenamed(
    "product_category_name_english",
    "product_category_name_en"
)

# Final check — how many rows, how many got translated
total = silver_products.count()
translated = silver_products.filter(
    F.col("product_category_name_en").isNotNull()
).count()
unknown = silver_products.filter(
    F.col("product_category_name") == "unknown"
).count()

print(f"Total products: {total}")
print(f"Successfully translated: {translated}")
print(f"Unknown category: {unknown}")

# Save
silver_products.write.mode("overwrite").parquet("silver/products")
print(f"Silver products saved.")

Total products: 32951
Successfully translated: 32328
Unknown category: 610
Silver products saved.


In [23]:
silver_products.filter(
    (F.col("product_category_name_en").isNull()) &
    (F.col("product_category_name") != "unknown")
).select(
    "product_category_name",
    "product_category_name_en"
).distinct().show()

+---------------------+------------------------+
|product_category_name|product_category_name_en|
+---------------------+------------------------+
|             pc_gamer|                    NULL|
| portateis_cozinha...|                    NULL|
+---------------------+------------------------+



In [24]:
# Manual enrichment applied for categories
# missing from the source translation table.

In [25]:
# Manual translation for categories missing from translation table
# Evidence: 2 categories exist in products but not in translation table
# pc_gamer and portateis_cozinha_e_preparadores_de_alimentos

manual_translations = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers"
}

silver_products = silver_products.withColumn(
    "product_category_name_en",
    F.when(
        F.col("product_category_name") == "pc_gamer",
        "pc_gamer"
    ).when(
        F.col("product_category_name") == "portateis_cozinha_e_preparadores_de_alimentos",
        "portable_kitchen_food_preparers"
    ).otherwise(F.col("product_category_name_en"))
)

# Verify
silver_products.filter(
    F.col("product_category_name").isin(
        "pc_gamer", 
        "portateis_cozinha_e_preparadores_de_alimentos"
    )
).select(
    "product_category_name",
    "product_category_name_en"
).distinct().show()

+---------------------+------------------------+
|product_category_name|product_category_name_en|
+---------------------+------------------------+
|             pc_gamer|                pc_gamer|
| portateis_cozinha...|    portable_kitchen_...|
+---------------------+------------------------+



In [26]:
silver_products.write.mode("overwrite").parquet("silver/products")
print(f"Silver products saved: {silver_products.count()} rows")

# Final verification
total = silver_products.count()
translated = silver_products.filter(F.col("product_category_name_en").isNotNull()).count()
still_null = total - translated

print(f"Total: {total}")
print(f"Translated: {translated}")
print(f"Still null (unknown category): {still_null}")

Silver products saved: 32951 rows
Total: 32951
Translated: 32341
Still null (unknown category): 610


In [27]:
profile_table(sellers, "sellers")


TABLE: SELLERS
Rows: 3095 | Columns: 4

Schema:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

Null counts:
+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|0        |0                     |0          |0           |
+---------+----------------------+-----------+------------+

Duplicate rows: 0

Sample records (3 rows):
+--------------------+----------------------+--------------+------------+
|           seller_id|seller_zip_code_prefix|   seller_city|seller_state|
+--------------------+----------------------+--------------+------------+
|3442f8959a84dea7e...|                 13023|      campinas|          SP|
|d1b65fc7debc3361e...|                 13844|    mogi guacu|          SP|
|ce3ad9de960102d06...|            

In [28]:
# sellers — No transformation required
# Profiling showed: 0 nulls, 0 duplicate rows, 4 columns
# City names consistently lowercase — standardized, not dirty
# 3,095 unique sellers across the platform

silver_sellers = sellers
silver_sellers.write.mode("overwrite").parquet("silver/sellers")
print(f"Silver sellers saved: {silver_sellers.count()} rows")

Silver sellers saved: 3095 rows


In [29]:
profile_table(order_payments, "order_payments")


TABLE: ORDER_PAYMENTS
Rows: 103886 | Columns: 5

Schema:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

Null counts:
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|0       |0                 |0           |0                   |0            |
+--------+------------------+------------+--------------------+-------------+

Duplicate rows: 0

Sample records (3 rows):
+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+

In [30]:
# Find orders with multiple payment rows
from pyspark.sql import functions as F

order_payments.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.col("count").desc()) \
    .show(5)

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|fa65dad1b0e818e3c...|   29|
|ccf804e764ed5650c...|   26|
|285c2e15bebd4ac83...|   22|
|895ab968e7bb0d565...|   21|
|ee9ca989fc93ba09a...|   19|
+--------------------+-----+
only showing top 5 rows


In [31]:
order_payments.filter(
    F.col("order_id") == "fa65dad1b0e818e3c..."
).orderBy("payment_sequential").show(30)

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
+--------+------------------+------------+--------------------+-------------+



In [32]:
multi_payment_orders = order_payments.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.col("count").desc())

# Get the actual full order_id
top_order_id = multi_payment_orders.first()["order_id"]
print(top_order_id)

# Now look at its payments
order_payments.filter(
    F.col("order_id") == top_order_id
).orderBy("payment_sequential").show(30)

fa65dad1b0e818e3ccc5cb0e39231352
+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|fa65dad1b0e818e3c...|                 1|     voucher|                   1|         3.71|
|fa65dad1b0e818e3c...|                 2|     voucher|                   1|         8.51|
|fa65dad1b0e818e3c...|                 3|     voucher|                   1|         2.95|
|fa65dad1b0e818e3c...|                 4|     voucher|                   1|        29.16|
|fa65dad1b0e818e3c...|                 5|     voucher|                   1|         0.66|
|fa65dad1b0e818e3c...|                 6|     voucher|                   1|         5.02|
|fa65dad1b0e818e3c...|                 7|     voucher|                   1|         0.32|
|fa65dad1b0e818e3c...|                 8|     voucher|             

In [33]:
# Flag zero value payments — not wrong, but worth knowing
silver_payments = order_payments.withColumn(
    "is_zero_value",
    F.when(F.col("payment_value") == 0.0, True).otherwise(False)
)

# Verify
print("Zero value payments:")
silver_payments.filter(F.col("is_zero_value") == True).count()

Zero value payments:


9

In [34]:
order_payments.select(
    F.count("*").alias("total"),
    F.countDistinct(
        "order_id",
        "payment_sequential"
    ).alias("distinct_combinations")
).show()

+------+---------------------+
| total|distinct_combinations|
+------+---------------------+
|103886|               103886|
+------+---------------------+



In [35]:
# order_payments — No transformation required  
# Profiling showed: 0 nulls, 0 duplicate rows
# Multiple rows per order = legitimate business behavior (multiple payment methods)
# 9 zero-value vouchers exist — explainable as expired/empty vouchers, not anomalies
# Preserving payment-level granularity for Gold layer

silver_payments = order_payments
silver_payments.write.mode("overwrite").parquet("silver/order_payments")
print(f"Silver order_payments saved: {silver_payments.count()} rows")

Silver order_payments saved: 103886 rows


In [36]:
profile_table(order_reviews, "order_reviews")


TABLE: ORDER_REVIEWS
Rows: 99224 | Columns: 7

Schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

Null counts:
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|0        |0       |0           |87656               |58247                 |0                   |0                      |
+---------+--------+------------+--------------------+----------------------+-----------

In [37]:
order_reviews.filter(
    F.col("order_id").isNull()
).show(10, truncate=False)

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [38]:
order_reviews.select(
    F.count("*").alias("rows"),
    F.countDistinct("review_id").alias("distinct_reviews")
).show()

+-----+----------------+
| rows|distinct_reviews|
+-----+----------------+
|99224|           98410|
+-----+----------------+



In [39]:
order_reviews.select(
    F.countDistinct("order_id")
).show()

+------------------------+
|count(DISTINCT order_id)|
+------------------------+
|                   98673|
+------------------------+



In [40]:
order_reviews.groupBy(
    order_reviews.columns
).count().filter(
    F.col("count") > 1
).show(20, truncate=False)

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|count|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+



In [41]:
order_reviews.filter(
    F.length("review_id") > 40
).select(
    "review_id"
).show(20, truncate=False)

+---------+
|review_id|
+---------+
+---------+



In [42]:
order_reviews.filter(
    F.col("review_id").rlike("[ ]")
).count()

0

In [43]:
profile_table(order_reviews, "order_reviews")


TABLE: ORDER_REVIEWS
Rows: 99224 | Columns: 7

Schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

Null counts:
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|0        |0       |0           |87656               |58247                 |0                   |0                      |
+---------+--------+------------+--------------------+----------------------+-----------

In [6]:
profile_table(order_reviews, "order_reviews")


TABLE: ORDER_REVIEWS
Rows: 99224 | Columns: 7

Schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

Null counts:
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|0        |0       |0           |87656               |58247                 |0                   |0                      |
+---------+--------+------------+--------------------+----------------------+-----------

In [7]:
order_reviews.groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .show()

+------------+-----+
|review_score|count|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19142|
|           5|57328|
+------------+-----+



In [8]:
# order_reviews — No transformation required
# Bronze ingestion was corrected for multiline review comments
# Profiling showed:
# - 0 null review_id
# - 0 null order_id
# - 0 duplicate rows
# - Valid timestamps for review creation and response
# Null review titles/messages are expected business behavior
# because many customers provide only a rating without text feedback.
# Preserving review-level granularity for Gold layer.

silver_reviews = order_reviews

silver_reviews.write.mode("overwrite").parquet("silver/order_reviews")

print(f"Silver order_reviews saved: {silver_reviews.count()} rows")

Silver order_reviews saved: 99224 rows


In [9]:
profile_table(sellers, "sellers")


TABLE: SELLERS
Rows: 3095 | Columns: 4

Schema:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

Null counts:
+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|0        |0                     |0          |0           |
+---------+----------------------+-----------+------------+

Duplicate rows: 0

Sample records (3 rows):
+--------------------+----------------------+--------------+------------+
|           seller_id|seller_zip_code_prefix|   seller_city|seller_state|
+--------------------+----------------------+--------------+------------+
|3442f8959a84dea7e...|                 13023|      campinas|          SP|
|d1b65fc7debc3361e...|                 13844|    mogi guacu|          SP|
|ce3ad9de960102d06...|            

In [10]:
# sellers — No transformation required
# Profiling showed:
# - 0 nulls
# - 0 duplicate rows
# - Consistent city/state information
# - seller_zip_code_prefix will be used for
#   geographic enrichment through the geolocation table
# Preserving as-is from Bronze.

silver_sellers = sellers

silver_sellers.write.mode("overwrite").parquet("silver/sellers")

print(f"Silver sellers saved: {silver_sellers.count()} rows")

Silver sellers saved: 3095 rows


In [11]:
profile_table(geolocation, "geolocation")


TABLE: GEOLOCATION
Rows: 1000163 | Columns: 5

Schema:
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

Null counts:
+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|0                          |0              |0              |0               |0                |
+---------------------------+---------------+---------------+----------------+-----------------+

Duplicate rows: 261831

Sample records (3 rows):
+---------------------------+------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|   geo

In [12]:
from pyspark.sql import functions as F

geolocation.groupBy("geolocation_zip_code_prefix") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(10)

+---------------------------+-----+
|geolocation_zip_code_prefix|count|
+---------------------------+-----+
|                      24220| 1146|
|                      24230| 1102|
|                      38400|  965|
|                      35500|  907|
|                      11680|  879|
|                      22631|  832|
|                      30140|  810|
|                      11740|  788|
|                      38408|  773|
|                      28970|  743|
+---------------------------+-----+
only showing top 10 rows


In [13]:
geolocation.filter(
    F.col("geolocation_zip_code_prefix") == 24220
).show(20, truncate=False)

+---------------------------+-------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng    |geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+----------------+-----------------+
|24220                      |-22.905816548704518|-43.106988856771984|niteroi         |RJ               |
|24220                      |-22.902305918691127|-43.11254462280002 |niteroi         |RJ               |
|24220                      |-22.904566594523498|-43.11049125088019 |niteroi         |RJ               |
|24220                      |-22.902574860407324|-43.109192038980986|niteroi         |RJ               |
|24220                      |-22.90749962932978 |-43.10616955390452 |niteroi         |RJ               |
|24220                      |-22.90431897584976 |-43.113001263950586|niteroi         |RJ               |
|24220                      |-22.899013761065042|-43.10

In [14]:
geolocation.select(
    F.count("*").alias("rows"),
    F.countDistinct("geolocation_zip_code_prefix").alias("distinct_prefixes")
).show()

+-------+-----------------+
|   rows|distinct_prefixes|
+-------+-----------------+
|1000163|            19015|
+-------+-----------------+



In [15]:
# geolocation — No transformation required
# Profiling showed: 0 nulls, 0 duplicate rows
# Multiple rows per zip prefix are expected business behavior
# Each row represents a valid geographic observation
# Preserving original granularity

silver_geolocation = geolocation

silver_geolocation.write.mode("overwrite").parquet("silver/geolocation")

print(
    f"Silver geolocation saved: "
    f"{silver_geolocation.count()} rows"
)

Silver geolocation saved: 1000163 rows


In [16]:
# geolocation — No transformation required
# Profiling showed: 0 nulls, 0 duplicate rows
# Multiple rows per zip code = legitimate geographic observations
# Each row is a valid coordinate reading, not a duplicate
# 
# Note for Gold layer: if joining customers to geolocation,
# aggregate geolocation first (avg lat/lng per zip code)
# within the Gold transformation — not here.
# Silver preserves the original observation-level granularity.

silver_geolocation = geolocation
silver_geolocation.write.mode("overwrite").parquet("silver/geolocation")
print(f"Silver geolocation saved: {silver_geolocation.count()} rows")

Silver geolocation saved: 1000163 rows
